# Task 2: Exploratory Data Analysis

Analyze Ethiopia Access and Usage patterns, infrastructure enablers, event timeline overlays, correlations, and data-quality limits.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from src.data_loader import load_unified_data, load_impact_sheet
from src.eda import summarize_dataset, temporal_coverage, growth_rates, sparse_indicators, indicator_correlations

FIG = ROOT / 'reports' / 'figures'
FIG.mkdir(parents=True, exist_ok=True)

data = load_unified_data()
impacts = load_impact_sheet()
obs = data[data.record_type=='observation'].copy()
obs['observation_date'] = pd.to_datetime(obs['observation_date'])
events = data[data.record_type=='event'].copy()
events['observation_date'] = pd.to_datetime(events['observation_date'])
summarize_dataset(data)


## 1. Dataset overview & data quality


In [ ]:
summary = summarize_dataset(data)
for k,v in summary.items():
    print(f'=== {k} ===')
    print(v, '\n')

cov = temporal_coverage(obs)
print(cov)
fig, ax = plt.subplots(figsize=(10,6))
sns.heatmap(cov, cmap='YlGnBu', cbar=True, ax=ax)
ax.set_title('Temporal coverage: indicator x year')
fig.tight_layout()
fig.savefig(FIG/'temporal_coverage.png', dpi=150)
plt.show()

print('Sparse indicators (<=2 points):')
print(sparse_indicators(obs))
print('\nConfidence distribution:')
print(obs.confidence.value_counts(dropna=False))


## 2. Access analysis (account ownership)


In [ ]:
access = obs[obs.indicator_code=='ACC_OWNERSHIP'].sort_values('observation_date')
gr = growth_rates(access)
print(gr)

fig = px.line(access, x='observation_date', y='value_numeric', markers=True,
              title='Ethiopia Account Ownership (Access), 2011-2024')
fig.update_yaxes(title='Account ownership (%)')
fig.show()
fig.write_html(FIG/'access_trajectory.html')

male = obs[obs.indicator_code=='ACC_OWNERSHIP_M']
female = obs[obs.indicator_code=='ACC_OWNERSHIP_F']
gender = pd.concat([male.assign(group='Male'), female.assign(group='Female')])
if len(gender):
    fig2 = px.bar(gender, x='observation_date', y='value_numeric', color='group', barmode='group',
                  title='Account ownership by gender')
    fig2.show()
    fig2.write_html(FIG/'gender_ownership.html')

urban = obs[obs.indicator_code=='ACC_OWNERSHIP_URBAN']
rural = obs[obs.indicator_code=='ACC_OWNERSHIP_RURAL']
if len(urban) and len(rural):
    print('Urban-rural gap 2024 (pp):', float(urban.value_numeric.iloc[-1]-rural.value_numeric.iloc[-1]))


## 3. Usage (digital payments) & registered-vs-survey gap


In [ ]:
usage = obs[obs.indicator_code=='USG_DIGITAL_PAYMENT'].sort_values('observation_date')
mm = obs[obs.indicator_code=='ACC_MM_ACCOUNT'].sort_values('observation_date')
fig = px.line(usage, x='observation_date', y='value_numeric', markers=True, title='Digital payment adoption (Usage)')
fig.show()
fig.write_html(FIG/'usage_trajectory.html')
print('MM account ownership points:')
print(mm[['observation_date','value_numeric']])
gap = obs[obs.indicator_code=='USG_REG_SURVEY_GAP']
print(gap[['observation_date','value_numeric','notes']])


## 4. Infrastructure / enablers


In [ ]:
infra_codes = ['ACC_4G_COV','ACC_MOBILE_PEN','AFF_DATA_INCOME','ACC_FAYDA']
infra = obs[obs.indicator_code.isin(infra_codes)]
fig = px.line(infra, x='observation_date', y='value_numeric', color='indicator_code', markers=True,
              title='Infrastructure and enabler indicators')
fig.show()
fig.write_html(FIG/'infrastructure_enablers.html')


## 5. Event timeline overlay


In [ ]:
fig = px.line(access, x='observation_date', y='value_numeric', markers=True, title='Access with event markers')
for _, e in events.iterrows():
    fig.add_vline(x=e.observation_date, line_dash='dot', opacity=0.4)
fig.show()
fig.write_html(FIG/'access_events_overlay.html')

timeline = events[['indicator','category','observation_date']].sort_values('observation_date')
fig_t = px.scatter(timeline, x='observation_date', y='category', color='category',
                   hover_name='indicator', title='Cataloged events timeline')
fig_t.show()
fig_t.write_html(FIG/'events_timeline.html')


## 6. Correlations & impact_link insights


In [ ]:
corr_codes = ['ACC_OWNERSHIP','USG_DIGITAL_PAYMENT','ACC_MM_ACCOUNT','ACC_4G_COV','GEN_GAP_ACC']
corr = indicator_correlations(obs, corr_codes)
print(corr)
fig, ax = plt.subplots(figsize=(7,5))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, ax=ax)
ax.set_title('Indicator correlations (yearly means)')
fig.tight_layout()
fig.savefig(FIG/'indicator_correlations.png', dpi=150)
plt.show()

print('Impact links by pillar:')
print(impacts.pillar.value_counts())
print(impacts.groupby(['parent_id','related_indicator'])['impact_estimate'].sum())


## 7. Key insights (≥5)

1. **Access decelerated sharply after 2021:** +11pp (2017–2021) then only **+3pp** (2021–2024) despite Telebirr/M-Pesa scale-up.
2. **Registered ≠ included:** operator registrations (Telebirr 54M+) dwarf survey MM ownership (9.45%) — dormant/duplicate accounts and definition gaps.
3. **Usage is rising faster than Access:** digital payment adoption climbs from low single digits historically to **35%** in 2024.
4. **Gender gap is narrowing but persistent:** ~20pp in 2021 vs a smaller male–female gap in 2024.
5. **Geography binds national averages:** urban ownership (~72%) far exceeds rural (~40%).
6. **P2P/ATM crossover** signals a payments-behavior shift even while Findex Access stalls.
7. **Data limits:** few Findex points, medium-confidence historical Usage anchors, sparse enabler time series.

See also `reports/eda_key_insights.md` and `reports/data_quality_assessment.md`.
